# 02 — Training (D3–D6)

**YOLOv8s / YOLO11s / YOLO26s × 3 seeds, identical settings.**

The paper's claim is *identical conditions*. Everything comes from `src/config.py`. **Do not tune per model** — a different lr for one model kills the controlled comparison and the contribution with it.

---
### Multi-seed protocol

Measured on T4: **~42 s/epoch true** (39 s train + 3 s val) → ~28 min per model per seed → **~4.2 h for the full 3×3 matrix**, against a ~30 h weekly quota.

**Why 3 seeds and not 1.** The central claim is a *comparison between architectures*. A single-seed gap of ~0.01 mAP sits inside run-to-run noise for detection, so single-seed results license no claim at all — a reviewer can dismiss the whole Axis B result as noise. With 3 seeds you report mean±std and the comparison becomes defensible.

Seeds are the **outer** loop deliberately: if the 12 h session cap interrupts you, you're left with complete seeds rather than a ragged matrix that can't be reported as mean±std. Finished runs are skipped, so re-running resumes cleanly.

---
### ⚠️ Accelerator: `GPU T4 x2` — NOT P100

P100 is **sm_60**; PyTorch requires **sm_70+**. It fails or silently falls back to CPU. Select **`GPU T4 x2`** (sm_75), keep `device=0`. T4 has FP16 tensor cores but **no bf16** — `amp=True`, never `bf16`.

**Don't attempt 2×T4 DDP.** Kaggle bills by wall-clock not per-GPU, so `T4 x2` with `device=0` costs the same quota.

---
### Prerequisite

Notebook 01 → Save Version → **"Save & Run All (Commit)"** (not Quick Save; push to GitHub first, the commit re-clones). Then here: **Add Data** → **"Notebook Output Files"** → **"Your Work"** → notebook 01.

In [ ]:
!pip install -q ultralytics wandb

REPO = "/kaggle/working/repo"
!rm -rf {REPO} && git clone -q https://github.com/ShMazumder/vinbigdata-chest-x-ray.git {REPO}

import sys
sys.path.insert(0, REPO)

from pathlib import Path
import pandas as pd

from src import config as C
from src.training import train as T
from src.utils.run_logger import RunLogger, benchmark_inference, capture_gpu

!cd {REPO} && git rev-parse --short HEAD
print(f"seeds: {C.SEEDS} | models: {list(C.MODELS)}")

In [ ]:
# strict=True -> raises immediately on sm_60 (P100) rather than after a session
# of confusing CPU-speed epochs.
gpu = capture_gpu(strict=True)
gpu

In [ ]:
DATA_YAML = C.find_data_yaml()
# Fallback if the search ever misses -- same validation runs:
# DATA_YAML = C.find_data_yaml(
#     "/kaggle/input/notebooks/<username>/01-data-prep/vindr_yolo/data.yaml")

### Stage locally

Read speed goes 35 MB/s → ~3000 MB/s and Ultralytics can finally write its label cache (saves ~15 s/run).

> Note: this does **not** speed up training. Measured epoch time is identical mounted vs staged — the dataloader workers were already hiding read latency behind GPU compute. You are GPU-bound. Staging is kept for the label cache and to silence the slow-access warning.

In [ ]:
DATA_YAML = C.stage_dataset_local(DATA_YAML)
DATA_ROOT = C.dataset_root(DATA_YAML)
print(DATA_YAML.read_text())

## 1. Smoke test + W&B overhead (~10 min)

5 epochs each way. Validates the whole training path before committing 4 h to it.

**Use `sec_per_epoch_true`**, parsed from Ultralytics' `results.csv`. The wall-clock figure divides by epochs and folds in setup — weight downloads, AMP check, label scan — which inflated a 5-epoch run by ~25% and produced a nonsensical *negative* W&B overhead on the first attempt.

In [ ]:
_ = T.train_one("yolov8s", DATA_YAML, epochs=5, use_wandb=False,
                runs_dir=C.RUNS_DIR / "smoke_nowandb")

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    have_wandb = True
except Exception as e:
    print(f"no W&B secret ({e}) -- continuing without. RunLogger still records everything.")
    have_wandb = False

if have_wandb:
    _ = T.train_one("yolov8s", DATA_YAML, epochs=5, use_wandb=True,
                    runs_dir=C.RUNS_DIR / "smoke_wandb")

In [ ]:
def rate(d):
    t = RunLogger.master_table(d).iloc[-1]
    return t.get("sec_per_epoch_true") or t["sec_per_epoch"]

a = rate(C.RUNS_DIR / "smoke_nowandb")
n_runs = len(C.MODELS) * len(C.SEEDS)
print(f"{a:.1f}s/epoch (true) -> {a*C.EPOCHS/60:.0f} min per run")
print(f"full matrix: {n_runs} runs -> {a*C.EPOCHS*n_runs/3600:.1f} h")

if have_wandb:
    b = rate(C.RUNS_DIR / "smoke_wandb")
    print(f"wandb overhead: {(b/a-1)*100:+.1f}%")

## 2. Full matrix — 3 models × 3 seeds (~4.2 h)

Checkpoints save every epoch; completed runs are skipped on re-entry. If the 12 h cap kills the session, just re-run this cell.

Two models instead of three: `models=["yolov8s", "yolo26s"]` — that pair is Axis B (NMS vs NMS-free), the contrast that ties the paper's halves together.

In [ ]:
results = T.train_all_seeds(DATA_YAML, use_wandb=have_wandb)
pd.Series({f"{k[0]}_s{k[1]}": v["status"] for k, v in results.items()})

## 3. Evaluation — mAP@0.4 across seeds

> ⚠️ **Ultralytics cannot give you mAP@0.4.** It reports @0.5 and @0.5:0.95 only; VinBigData used IoU > 0.4. If @0.5 lands in a column labelled @0.4, the related-work comparison misaligns and the error survives to camera-ready.

> [!CAUTION]
> **Our numbers are not directly comparable to published ones.** We train and evaluate on the **positive-only subset** — every image contains a finding, so the model never has an opportunity to false-positive on a healthy chest. That inflates precision and mAP relative to a full-set evaluation.
>
> Evidence: the 5-epoch smoke test already reached **mAP@0.5 = 0.310** against YOLO-CXR's published **0.338** at full training. A 5-epoch model does not nearly match a published architecture — the protocols differ.
>
> Keep published figures in a *related work* table with an explicit protocol column. State the protocol in the **abstract**, not just methods.

In [ ]:
from ultralytics import YOLO
from src.evaluation import metrics as M

weights = T.all_weights_by_seed()
evals = {}
for (key, seed), w in weights.items():
    print(f"\n=== {key} seed={seed} ===")
    evals[(key, seed)] = M.evaluate_full(
        YOLO(w), DATA_ROOT / "images" / "test", DATA_ROOT / "labels" / "test",
        imgsz=C.IMGSZ,
    )

In [ ]:
agg = M.aggregate_seeds(evals)
print(agg[["mAP@0.4", "mAP@0.5", "n_seeds"]])
M.seed_spread_warning(agg)

> **If the warning says the gap is inside seed noise, that is not a failure.**
>
> "Three modern YOLO architectures are statistically indistinguishable on VinDr-CXR detection, but differ measurably in explanation faithfulness" is a *stronger* paper than a spurious 0.008 mAP ranking. It sharpens the argument that accuracy is not the interesting axis — which is the thesis.
>
> Report the models as indistinguishable and let Axis B carry the comparison.

### ⛔ Gate D4

Reference reached ~0.45 CV positive-only with EfficientDet at 896–1024px, 50 epochs. You're at 512px, `s`-scale, 40 epochs.

- **0.25–0.40** — reasonable, proceed
- **< 0.20** — labels or fusion are broken, not the model. Stop.
- **> 0.50** — check the split for leakage before believing it.

In [ ]:
m = agg.loc["yolov8s", "map40_mean"]
if m < 0.20:
    print(f"⛔ GATE FAILED: mAP@0.4={m:.4f} < 0.20. Debug the data pipeline.")
elif m > 0.50:
    print(f"⚠ mAP@0.4={m:.4f} unexpectedly high. Check the split for leakage.")
else:
    print(f"✓ gate passed: mAP@0.4={m:.4f}")

### Per-class — where the score actually comes from

Smoke test (5 epochs, yolov8s):

| | mAP@0.5 |
|---|---|
| Aortic enlargement | **0.945** |
| Cardiomegaly | **0.912** |
| *other 12, mean* | **0.207** |
| all 14 | 0.310 |

Both leaders are **anatomically anchored** — the aorta and heart sit in the same place in every image, so a model can score by learning *position* rather than pathology. Strip them and real abnormality detection is ~0.21.

That is a finding, and it sets up **Axis A**: Calcification 0.097, Other lesion 0.045, against 0.945 for a fixed-location structure. **Report per-class throughout** — a single mAP hides the whole story.

In [ ]:
# Per-class mean across seeds, with n_gt.
pc_rows = []
for (key, seed), ev in evals.items():
    for cls, ap in ev["detail_40"]["per_class"].items():
        pc_rows.append({"model": key, "seed": seed, "class": cls, "ap40": ap})
pc = pd.DataFrame(pc_rows)

table = pc.pivot_table(index="class", columns="model", values="ap40",
                       aggfunc=["mean", "std"]).round(4)
n_gt = pd.Series(list(evals.values())[0]["detail_40"]["n_gt"], name="n_gt")
table = table.join(n_gt).sort_values("n_gt")
print(table)

anchored = ["Aortic enlargement", "Cardiomegaly"]
for key in pc.model.unique():
    s = pc[pc.model == key].groupby("class").ap40.mean().dropna()
    print(f"{key}: all-14 {s.mean():.4f} | "
          f"excl. anchored {s.drop(anchored, errors='ignore').mean():.4f}")

> **Pneumothorax (n≈10) will look degenerate.** The smoke test gave `P=1.000, R=0.000` — one near-miss and nothing else. That is what AP on ten boxes looks like. Reporting rule was fixed in notebook 01 before any score was seen: keep all 14 in mAP, show `n`, caveat n<30 in limitations.

## 4. Inference benchmark (D10) — same card, one session

> ⚠️ **mAP is hardware-independent. Latency is not.** YOLO26 is marketed as edge-first (claimed 43% CPU speed gain over YOLO11), so a latency table is expected. Valid only if produced **here** — all checkpoints back to back, one session, one card.

Benchmarks seed 0 only; latency doesn't vary by seed.

In [ ]:
RunLogger.check_hardware_consistency(C.RUNS_DIR)
seed0 = {k: v for (k, s), v in weights.items() if s == C.SEEDS[0]}
bench = benchmark_inference(seed0, imgsz=C.IMGSZ, out_dir=str(C.RUNS_DIR))
pd.DataFrame(bench).T[["ms_per_image", "fps", "n_params_M", "gpu_family"]]

## 5. Persist

`/kaggle/working` dies with the session. Commit via **Save Version → "Save & Run All"** so notebook 03 can attach these weights.

In [ ]:
import json
(C.RUNS_DIR / "eval_summary.json").write_text(json.dumps(
    {f"{k[0]}|seed{k[1]}": {
        "model": k[0], "seed": k[1],
        "map40": v["map40"], "map50": v["map50"],
        "per_class_40": v["detail_40"]["per_class"],
        "n_gt": v["detail_40"]["n_gt"]} for k, v in evals.items()},
    indent=2, default=str))
agg.to_csv(C.RUNS_DIR / "agg_by_model.csv")
table.to_csv(C.RUNS_DIR / "per_class_by_model.csv")

print(f"{len(weights)} runs logged")
print("Next: Save Version -> 'Save & Run All (Commit)', then attach to notebook 03.")